# Actividad 5: MLOps, Experiment Tracking con MLflow y Comparación de Modelos
### Máster en Inteligencia Artificial | Curso: Gestión de Proyectos en IA

---
### 📌 Objetivo de la Actividad:
1. Diseñar e implementar un flujo modular de Machine Learning con buenas prácticas de ingeniería de software.
2. Comparar al menos dos algoritmos de clasificación (Regresión Logística vs. Random Forest vs. XGBoost).
3. Optimizar hiperparámetros mediante validación cruzada estratificada (Stratified 5-Fold Grid Search).
4. Registrar parámetros, métricas estandarizadas (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Log Loss) y artefactos en **MLflow**.
5. Evaluar trade-offs, varianza entre folds y reflexionar sobre deuda técnica y gobierno de modelos en producción.

**Dataset:** Telco Customer Churn (IBM) — 7,043 registros, 20 variables de clientes y comportamiento.


## 1. Configuración de Dependencias y Entorno
En Google Colab, descomente la siguiente celda para instalar las dependencias requeridas.


In [ ]:
# Si se ejecuta en Google Colab:
# !pip install -q mlflow xgboost scikit-learn pandas matplotlib seaborn

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Entorno inicializado correctamente.")


## 2. Inicialización y Configuración de MLflow Tracking
Configuramos el seguimiento de experimentos. MLflow registrará los hiperparámetros, métricas fold a fold, métricas finales en test, matrices de confusión, curvas ROC y los modelos serializados con sus respectivas firmas (*model signatures*).


In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

# Usar backend SQLite local para máxima compatibilidad con MLflow 3.x
tracking_uri = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(tracking_uri)

nombre_experimento = "Telco_Customer_Churn_Optimization"
mlflow.set_experiment(nombre_experimento)

print(f"Tracking URI activo: {tracking_uri}")
print(f"Experimento MLflow:  {nombre_experimento}")


## 3. Carga de Datos y Análisis Exploratorio (EDA)
Descargamos el dataset oficial crudo desde el repositorio de IBM.


In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df_raw = pd.read_csv(url)

print(f"Forma original: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
display(df_raw.head(3))


In [ ]:
# Inspección de calidad inicial
print("Valores nulos estándar por columna:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

# Detección de anomalía conocida: espacios en blanco en TotalCharges
espacios_blanco = (df_raw['TotalCharges'].astype(str).str.strip() == '').sum()
print(f"\nEspacios en blanco detectados en 'TotalCharges': {espacios_blanco}")

# Distribución de la variable objetivo (Churn)
churn_prop = df_raw['Churn'].value_counts(normalize=True)
print("\nProporción de Churn (Target):")
print(churn_prop)


## 4. Limpieza de Datos y Reglas de Negocio
Se aplican las transformaciones del módulo `datos_prep.py`:
1. **Eliminación de `customerID`**: Identificador arbitrario sin señal predictiva.
2. **Regla de negocio en `TotalCharges`**: Los 11 registros en blanco corresponden exactamente a clientes con `tenure == 0` (clientes en su mes de alta sin cobros acumulados). Se imputan con `0.0`.
3. **Conversión y codificación binaria**: La variable objetivo `Churn` se codifica como 1 (abandono) y 0 (permanencia).


In [ ]:
# Limpieza rigurosa guiada por negocio
df_clean = df_raw.copy()

# 1. Descartar identificador
if 'customerID' in df_clean.columns:
    df_clean = df_clean.drop(columns=['customerID'])

# 2. Imputación por regla de negocio
df_clean['TotalCharges'] = df_clean['TotalCharges'].replace(r'^\s*$', np.nan, regex=True)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'])
df_clean.loc[(df_clean['tenure'] == 0) & (df_clean['TotalCharges'].isna()), 'TotalCharges'] = 0.0

# 3. Codificación de target
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

# 4. Asegurar tipos numéricos
df_clean['tenure'] = df_clean['tenure'].astype(int)
df_clean['MonthlyCharges'] = df_clean['MonthlyCharges'].astype(float)
df_clean['TotalCharges'] = df_clean['TotalCharges'].astype(float)

print(f"Forma tras limpieza: {df_clean.shape}")
print(f"Nulos restantes: {df_clean.isna().sum().sum()}")


## 5. Partición y Pipeline de Preprocesamiento
Para evitar el **data leakage**, encapsulamos las transformaciones en un `ColumnTransformer`:
- `StandardScaler` para las 3 variables numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`).
- `OneHotEncoder(drop='first', handle_unknown='ignore')` para las 16 variables categóricas.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

# Partición estratificada 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_num),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cols_cat)
    ]
)

print(f"Conjunto de Entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Conjunto de Prueba:        X={X_test.shape}, y={y_test.shape}")


## 6. Entrenamiento, Ajuste de Hiperparámetros y MLflow Tracking
Entrenamos y evaluamos 3 algoritmos con validación cruzada estratificada de 5 pliegues (`StratifiedKFold`), optimizando la métrica `ROC-AUC`:
1. **Regresión Logística**
2. **Random Forest Classifier**
3. **XGBoost Classifier**


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_config = {
    "Regresion_Logistica": {
        "estimator": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
        "params": {
            'classifier__C': [0.01, 0.1, 1.0, 10.0],
            'classifier__penalty': ['l1', 'l2']
        }
    },
    "Random_Forest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [5, 10, None],
            'classifier__min_samples_split': [2, 5]
        }
    },
    "XGBoost": {
        "estimator": XGBClassifier(eval_metric='logloss', random_state=42),
        "params": {
            'classifier__n_estimators': [100, 150],
            'classifier__max_depth': [3, 5],
            'classifier__learning_rate': [0.05, 0.1]
        }
    }
}

resultados = []
probabilidades = {}

for nombre, cfg in modelos_config.items():
    print(f"Optimizando {nombre}...")
    pipe = Pipeline([
        ('preprocessor', preprocesador),
        ('classifier', cfg['estimator'])
    ])
    
    gs = GridSearchCV(pipe, cfg['params'], cv=cv, scoring='roc_auc', n_jobs=-1, return_train_score=True)
    gs.fit(X_train, y_train)
    best_pipe = gs.best_estimator_
    
    # Inferencia en prueba
    y_pred = best_pipe.predict(X_test)
    y_prob = best_pipe.predict_proba(X_test)[:, 1]
    probabilidades[nombre] = y_prob
    
    # Cálculo de métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    ll = log_loss(y_test, y_prob)
    
    cv_mean = gs.cv_results_['mean_test_score'][gs.best_index_]
    cv_std = gs.cv_results_['std_test_score'][gs.best_index_]
    
    # Registro en MLflow
    with mlflow.start_run(run_name=f"{nombre}_Notebook"):
        mlflow.set_tag("algoritmo", nombre)
        mlflow.set_tag("entorno", "Jupyter_Notebook")
        for p, v in gs.best_params_.items():
            mlflow.log_param(p, v)
        mlflow.log_metric("cv_mean_roc_auc", cv_mean)
        mlflow.log_metric("cv_std_roc_auc", cv_std)
        mlflow.log_metric("test_roc_auc", roc_auc)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_precision", prec)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_pr_auc", pr_auc)
        mlflow.log_metric("test_log_loss", ll)
        
        sample_in = X_test.head(3)
        sig = infer_signature(sample_in, best_pipe.predict(sample_in))
        mlflow.sklearn.log_model(
            sk_model=best_pipe,
            name="modelo_empaquetado",
            signature=sig,
            input_example=sample_in,
            serialization_format="cloudpickle"
        )
    
    resultados.append({
        "Modelo": nombre,
        "CV ROC-AUC": round(cv_mean, 4),
        "CV Desv": round(cv_std, 4),
        "Test ROC-AUC": round(roc_auc, 4),
        "Test F1-Score": round(f1, 4),
        "Test Accuracy": round(acc, 4),
        "Test Precision": round(prec, 4),
        "Test Recall": round(rec, 4),
        "Mejores Params": str(gs.best_params_)
    })
    print(f"  -> {nombre}: Test ROC-AUC = {roc_auc:.4f} | F1 = {f1:.4f} | Acc = {acc:.4f}")


## 7. Comparación Estadística y Gráficas de Rendimiento
Visualizamos las métricas y comparamos las curvas ROC.


In [ ]:
df_metricas = pd.DataFrame(resultados)
print("TABLA COMPARATIVA DE DESEMPEÑO:")
display(df_metricas[['Modelo', 'CV ROC-AUC', 'Test ROC-AUC', 'Test F1-Score', 'Test Accuracy', 'Test Recall', 'Test Precision']])

# Curvas ROC
plt.figure(figsize=(8, 6))
colores = {'Regresion_Logistica': '#1f77b4', 'Random_Forest': '#2ca02c', 'XGBoost': '#d62728'}
for nombre, prob in probabilidades.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_score = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{nombre} (AUC = {auc_score:.4f})", color=colores[nombre], lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.title('Comparativa de Curvas ROC en Conjunto de Prueba', fontsize=13, fontweight='bold')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR / Recall)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()


## 8. Reflexión sobre Deuda Técnica y Estrategia MLOps

### 1. Compensaciones Técnicas (Trade-offs):
- **XGBoost:** Alcanza el mayor ROC-AUC (**0.8464**), lo que optimiza el ordenamiento probabilístico de propensión al abandono. Es el modelo más potente para campañas segmentadas por deciles.
- **Regresión Logística:** Ofrece el mejor balance F1 (**0.6040**) con umbral por defecto (0.50) y máxima interpretabilidad analítica (pesos $\beta$).
- **Random Forest:** Tiende a un comportamiento conservador con mayor precisión (**0.6740**) y menor recall (**0.4091**).

### 2. Deuda Técnica en Producción:
- **Deriva de Datos (*Data Drift*):** Los hábitos de consumo de telecomunicaciones varían ante cambios de tarifas de la competencia o estacionalidad. Se recomienda monitoreo continuo mediante métricas KS (*Kolmogorov-Smirnov*) y Population Stability Index (PSI).
- **Sesgo de Inferencia (*Train-Serving Skew*):** Evitado mediante el encapsulamiento de transformaciones en `Pipeline`, asegurando que toda transformación aplicada en entrenamiento se replique idénticamente en inferencia.
- **Linaje y Reproducibilidad:** Centralizado con **MLflow Tracking** y serialización de modelos con firma explícita de entradas y salidas.

### 3. Recomendación de Negocio:
Desplegar **XGBoost** en el Model Registry de MLflow y ajustar el umbral de decisión en función de la matriz de costo-beneficio de la campaña de retención de clientes.
